# GitNexus Agent - Ask Questions About Your Codebase

This notebook demonstrates an agentic structure that uses OpenAI as the LLM and GitNexus tools to answer questions about the microservices-demo codebase.

**Architecture:**
- LLM: OpenAI (gpt-4o or gpt-4)
- Tools: GitNexus (query, context, impact, cypher)
- Agent: Simple tool-calling loop with reasoning

## Setup & Configuration

In [4]:
import subprocess
import json
import os
from pathlib import Path
from typing import Any, Optional
from dotenv import load_dotenv
from openai import OpenAI

# Load environment variables from .env file
load_dotenv()

# Configure OpenAI
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY not found. Please set it in .env file or as an environment variable.")

client = OpenAI(api_key=api_key)

print(f"✓ OpenAI API key loaded")
print(f"✓ Using model: gpt-4o-mini")

# ============================================================================
# SET YOUR REPOSITORY PATH HERE
# Leave as None to use the current working directory.
# ============================================================================
REPO_DIR = Path(r"C:\Users\EwegConsulting\Documents\Projects\gitnexus docker\gitnexus\repos\microservices-demo") # e.g. Path(r"C:\Projects\my-repo") or Path("/home/user/my-repo")
# ============================================================================

if REPO_DIR is None:
    REPO_DIR = Path.cwd()
else:
    REPO_DIR = Path(REPO_DIR)
    if not REPO_DIR.exists():
        raise ValueError(f"REPO_DIR does not exist: {REPO_DIR}")

print(f"✓ Working directory: {REPO_DIR}")

# Verify GitNexus connection
result = subprocess.run(
    "gitnexus status",
    shell=True,
    capture_output=True,
    text=True,
    cwd=REPO_DIR
)
print(f"GitNexus Status: {result.stdout}")
if result.returncode != 0:
    print(f"Warning: {result.stderr}")


✓ OpenAI API key loaded
✓ Using model: gpt-4o-mini
✓ Working directory: C:\Users\EwegConsulting\Documents\Projects\gitnexus docker\gitnexus\repos\microservices-demo
GitNexus Status: Repository: C:\Users\EwegConsulting\Documents\Projects\gitnexus docker\gitnexus\repos\microservices-demo
Indexed: 14-4-2026, 11:28:52
Indexed commit: c9857ee
Current commit: c9857ee
Status: âœ… up-to-date



## GitNexus Tool Wrappers

In [5]:
# Tool definitions for OpenAI function calling - all 7 official GitNexus MCP tools
TOOL_DEFINITIONS = [
    {
        "type": "function",
        "function": {
            "name": "gitnexus_query",
            "description": "Hybrid search (BM25 + semantic) to find execution flows and symbols related to a concept. Use this first when you don't know exact symbol names — returns process-grouped results ranked by relevance.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Concept, symptom, or feature to search for (e.g. 'payment processing', 'checkout flow', 'error handling')"
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "gitnexus_context",
            "description": "360-degree view of a specific code symbol: all incoming callers, outgoing callees, and which execution flows it participates in.",
            "parameters": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "description": "Symbol name (e.g. 'PlaceOrder', 'chargeCard', 'PaymentService')"
                    }
                },
                "required": ["name"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "gitnexus_impact",
            "description": "Blast radius analysis — what breaks or is affected if you modify a symbol. Returns dependents grouped by depth: d=1 WILL BREAK (direct callers), d=2 LIKELY AFFECTED, d=3 MAY NEED TESTING.",
            "parameters": {
                "type": "object",
                "properties": {
                    "target": {
                        "type": "string",
                        "description": "Symbol name to analyze"
                    },
                    "direction": {
                        "type": "string",
                        "enum": ["upstream", "downstream"],
                        "description": "upstream: who depends on this (callers); downstream: what this depends on (callees)"
                    },
                    "minConfidence": {
                        "type": "number",
                        "description": "Minimum confidence threshold 0-1 (default 0.7)"
                    },
                    "maxDepth": {
                        "type": "number",
                        "description": "Maximum relationship depth to traverse (default 3)"
                    }
                },
                "required": ["target", "direction"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "gitnexus_detect_changes",
            "description": "Maps current git-diff changes to affected symbols and execution flows. Use before committing to verify scope and assess risk.",
            "parameters": {
                "type": "object",
                "properties": {
                    "scope": {
                        "type": "string",
                        "enum": ["staged", "all", "unstaged", "compare"],
                        "description": "Which changes to analyze: staged, all (staged+unstaged), unstaged, or compare (vs a base ref)"
                    },
                    "base_ref": {
                        "type": "string",
                        "description": "Git reference for comparison (only when scope='compare', e.g. 'main', 'HEAD~1')"
                    }
                },
                "required": ["scope"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "gitnexus_rename",
            "description": "Coordinated multi-file symbol rename using the call graph. Produces confidence-tagged edits across all files. Always run with dry_run=true first to preview.",
            "parameters": {
                "type": "object",
                "properties": {
                    "symbol_name": {
                        "type": "string",
                        "description": "Current name of the symbol to rename"
                    },
                    "new_name": {
                        "type": "string",
                        "description": "Desired new name"
                    },
                    "dry_run": {
                        "type": "boolean",
                        "description": "If true, preview changes without applying them (default true)"
                    }
                },
                "required": ["symbol_name", "new_name"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "gitnexus_cypher",
            "description": "Execute a raw Cypher query against the code knowledge graph. Relationships use CodeRelation with a .type property (CALLS, IMPORTS, EXTENDS, IMPLEMENTS, DEFINES, MEMBER_OF, STEP_IN_PROCESS).",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Cypher query. Correct relationship syntax: MATCH (caller)-[:CodeRelation {type: 'CALLS'}]->(f:Function {name: 'X'}) RETURN caller.name, caller.filePath"
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "gitnexus_list_repos",
            "description": "List all repositories indexed by GitNexus.",
            "parameters": {
                "type": "object",
                "properties": {}
            }
        }
    }
]

print(f"Tool definitions created ({len(TOOL_DEFINITIONS)} tools)")
print(f"Available tools: {[t['function']['name'] for t in TOOL_DEFINITIONS]}")

Tool definitions created (7 tools)
Available tools: ['gitnexus_query', 'gitnexus_context', 'gitnexus_impact', 'gitnexus_detect_changes', 'gitnexus_rename', 'gitnexus_cypher', 'gitnexus_list_repos']


In [6]:
# Implement the GitNexus tool wrappers
def gitnexus_context(name):
    """
    Get detailed context about a code symbol.
    Shows: where it's defined, all callers, all callees, process participation.
    """
    result = subprocess.run(
        f'gitnexus context "{name}"',
        shell=True,
        capture_output=True,
        text=True,
        cwd=REPO_DIR
    )
    if result.returncode != 0:
        return {"error": f"Command failed: {result.stderr}"}
    try:
        return json.loads(result.stdout)
    except json.JSONDecodeError:
        return {"output": result.stdout}

def gitnexus_impact(target, direction="upstream", minConfidence=None, maxDepth=None):
    """
    Analyze the blast radius of changes to a code symbol.
    - direction='upstream': Who calls this? (what breaks if I change it?)
    - direction='downstream': What does this call?
    """
    cmd = f'gitnexus impact "{target}" --direction {direction}'
    if minConfidence is not None:
        cmd += f' --min-confidence {minConfidence}'
    if maxDepth is not None:
        cmd += f' --max-depth {maxDepth}'
    result = subprocess.run(
        cmd,
        shell=True,
        capture_output=True,
        text=True,
        cwd=REPO_DIR
    )
    if result.returncode != 0:
        return {"error": f"Command failed: {result.stderr}"}
    try:
        return json.loads(result.stdout)
    except json.JSONDecodeError:
        return {"output": result.stdout}

def gitnexus_cypher(query):
    """
    Execute a raw Cypher query against the code knowledge graph.
    Correct relationship syntax:
      MATCH (caller)-[:CodeRelation {type: 'CALLS'}]->(f:Function {name: 'X'})
      RETURN caller.name, caller.filePath
    Do NOT use CONTAINS on node properties (e.g. f.content CONTAINS '...').
    """
    result = subprocess.run(
        f'gitnexus cypher "{query}"',
        shell=True,
        capture_output=True,
        text=True,
        cwd=REPO_DIR
    )
    if result.returncode != 0:
        return {"error": f"Command failed: {result.stderr}"}
    try:
        return json.loads(result.stdout)
    except json.JSONDecodeError:
        return {"output": result.stdout}

def gitnexus_detect_changes(scope="staged", base_ref=None):
    """
    Map current git changes to affected symbols and execution flows.
    scope: 'staged', 'all', 'unstaged', or 'compare'
    base_ref: git reference for comparison (only with scope='compare')
    """
    cmd = f'gitnexus detect-changes --scope {scope}'
    if base_ref:
        cmd += f' --base-ref {base_ref}'
    result = subprocess.run(
        cmd,
        shell=True,
        capture_output=True,
        text=True,
        cwd=REPO_DIR
    )
    if result.returncode != 0:
        return {"error": f"Command failed: {result.stderr}"}
    try:
        return json.loads(result.stdout)
    except json.JSONDecodeError:
        return {"output": result.stdout}

def gitnexus_rename(symbol_name, new_name, dry_run=True):
    """
    Coordinated multi-file symbol rename using the call graph.
    Always run with dry_run=True first to preview changes.
    """
    cmd = f'gitnexus rename "{symbol_name}" "{new_name}"'
    if dry_run:
        cmd += ' --dry-run'
    result = subprocess.run(
        cmd,
        shell=True,
        capture_output=True,
        text=True,
        cwd=REPO_DIR
    )
    if result.returncode != 0:
        return {"error": f"Command failed: {result.stderr}"}
    try:
        return json.loads(result.stdout)
    except json.JSONDecodeError:
        return {"output": result.stdout}

def gitnexus_list_repos():
    """List all repositories indexed by GitNexus."""
    result = subprocess.run(
        'gitnexus list',
        shell=True,
        capture_output=True,
        text=True,
        cwd=REPO_DIR
    )
    if result.returncode != 0:
        return {"error": f"Command failed: {result.stderr}"}
    try:
        return json.loads(result.stdout)
    except json.JSONDecodeError:
        return {"output": result.stdout}

print("GitNexus tool wrappers implemented:")
print("  - gitnexus_context(name)")
print("  - gitnexus_impact(target, direction, minConfidence=None, maxDepth=None)")
print("  - gitnexus_cypher(query)")
print("  - gitnexus_detect_changes(scope='staged'|'all'|'unstaged'|'compare', base_ref=None)")
print("  - gitnexus_rename(symbol_name, new_name, dry_run=True)")
print("  - gitnexus_list_repos()")

GitNexus tool wrappers implemented:
  - gitnexus_context(name)
  - gitnexus_impact(target, direction, minConfidence=None, maxDepth=None)
  - gitnexus_cypher(query)
  - gitnexus_detect_changes(scope='staged'|'all'|'unstaged'|'compare', base_ref=None)
  - gitnexus_rename(symbol_name, new_name, dry_run=True)
  - gitnexus_list_repos()


In [7]:
# Also implement gitnexus_query for concept-based searches
def gitnexus_query(query):
    """
    Search the code graph by concept using hybrid BM25 + semantic search.
    Returns execution flows and symbols related to the search query.
    Best for: "find code about X" when you don't know exact symbol names.
    """
    result = subprocess.run(
        f'gitnexus query "{query}"',
        shell=True,
        capture_output=True,
        text=True,
        cwd=REPO_DIR
    )
    if result.returncode != 0:
        return {"error": f"Command failed: {result.stderr}"}
    try:
        return json.loads(result.stdout)
    except json.JSONDecodeError:
        return {"output": result.stdout}

print("✓ gitnexus_query(query) - concept-based search also implemented")

✓ gitnexus_query(query) - concept-based search also implemented


## Agent Implementation

In [8]:
# System prompt for the GitNexus code analysis agent
SYSTEM_PROMPT = """You are an expert code analysis agent with access to GitNexus tools that query a knowledge graph of the microservices-demo codebase (3477 symbols, 5994 relationships, 33 execution flows).

## Available Tools

### gitnexus_query(query)
Hybrid BM25 + semantic search. Returns execution flows and symbols grouped by process, ranked by relevance.
Use this first to orient yourself - it finds the right symbol names and related processes.
Examples: query("payment processing"), query("checkout flow"), query("error handling retry")

### gitnexus_context(name)
360-degree view of a specific symbol: all incoming callers, outgoing callees, and which execution flows it participates in.
Returns filePath, startLine, and endLine for each symbol - use these to locate source code.
If the result has status "ambiguous", the response includes a candidates list with full UIDs.
Re-call immediately with the full UID from that list to resolve it (e.g. "Method:src/checkoutservice/main.go:PlaceOrder").
Examples: context("chargeCard"), context("PlaceOrder"), context("Method:src/checkoutservice/main.go:PlaceOrder")

### gitnexus_impact(target, direction, minConfidence?, maxDepth?)
Blast radius analysis. Returns dependents grouped by depth with confidence scores:
  - d=1 -> WILL BREAK (direct callers/importers)
  - d=2 -> LIKELY AFFECTED (indirect dependencies)
  - d=3 -> MAY NEED TESTING (transitive effects)
Risk levels: <5 symbols/few processes = LOW | 5-15 symbols/2-5 processes = MEDIUM | >15 symbols = HIGH | auth/payments critical path = CRITICAL

### gitnexus_detect_changes(scope, base_ref?)
Maps current git-diff changes to affected symbols and execution flows. Use before committing.
scope: "staged" | "all" | "unstaged" | "compare"

### gitnexus_rename(symbol_name, new_name, dry_run?)
Coordinated multi-file rename using the call graph. Always use dry_run=True first to preview.

### gitnexus_cypher(query)
Raw Cypher queries against the knowledge graph for custom structural analysis.

Graph schema:
  Nodes: File, Function, Class, Interface, Method, Community, Process
  Edges: CodeRelation with .type property

Correct relationship traversal syntax:
  MATCH (caller)-[:CodeRelation {type: 'CALLS'}]->(f:Function {name: "chargeCard"})
  RETURN caller.name, caller.filePath

Reading source code of a file:
  MATCH (f:File {name: "server.js"}) RETURN f.content
  MATCH (f:File) WHERE f.filePath CONTAINS "cartservice" RETURN f.name, f.content
  Note: use RETURN f.content to read source - do NOT use f.content CONTAINS '...' in a WHERE clause (causes runtime error).

Use Cypher for enumeration, property lookups, structural queries, and reading file source.

### gitnexus_list_repos()
List all repositories indexed by GitNexus.

## Recommended Workflows

**"Which services depend on X / what would break?"**
1. gitnexus_query("X concept") -> find symbol names and related execution flows
2. gitnexus_context("X") -> confirm the symbol, see direct callers
3. gitnexus_impact("X", "upstream") -> full blast radius with depth and risk
4. gitnexus_context(each d=1 caller) -> understand exactly how they use X

**"How does flow Y work?"**
1. gitnexus_query("Y concept") -> find the processes and key symbols
2. gitnexus_context(key_symbol) -> drill into each symbol in the flow
3. gitnexus_cypher(...) for any structural queries needed

**"Is it safe to change X?"**
1. gitnexus_impact("X", "upstream") -> blast radius
2. Assess risk level from depth/symbol counts
3. gitnexus_detect_changes() for pre-commit verification

**"Rename / refactor X safely"**
1. gitnexus_rename(old, new, dry_run=True) -> preview all affected files
2. gitnexus_rename(old, new, dry_run=False) -> apply
3. gitnexus_detect_changes(scope="all") -> verify scope

**"Read the actual source code of X"**
1. gitnexus_context("X") -> get filePath and startLine/endLine
2. gitnexus_cypher("MATCH (f:File {name: 'filename'}) RETURN f.content") -> fetch full file source

## Key Principle
Always start with gitnexus_query to find correct symbol names. Never guess names - wrong symbols waste iterations. Then use context and impact to build a complete picture.
"""

print("System prompt defined")
print("Tools covered: query, context, impact, detect_changes, rename, cypher, list_repos")


System prompt defined
Tools covered: query, context, impact, detect_changes, rename, cypher, list_repos


In [9]:
from dataclasses import dataclass
import json

@dataclass
class Message:
    role: str
    content: str | list = ""


class CodebaseAgent:
    """Agent that uses OpenAI to answer questions about the codebase using GitNexus tools"""
    
    def __init__(self, model: str = "gpt-4o-mini", max_iterations: int = 25, max_result_chars: int = 5000):
        self.model = model
        self.max_iterations = max_iterations
        self.max_result_chars = max_result_chars
        self.conversation_history = []
        
    def run(self, question: str) -> str:
        self.conversation_history = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": question}
        ]
        
        separator = "=" * 70
        print()
        print(separator)
        print(f"Question: {question}")
        print(f"Model: {self.model}")
        print(separator)
        print()
        
        for iteration in range(self.max_iterations):
            print(f"[Iteration {iteration + 1}] Calling OpenAI...")
            
            try:
                response = client.chat.completions.create(
                    model=self.model,
                    messages=self.conversation_history,
                    tools=TOOL_DEFINITIONS,
                    tool_choice="auto",
                    temperature=0.5
                )
            except Exception as e:
                print(f"ERROR calling OpenAI: {e}")
                return f"Error: {e}"
            
            assistant_message = response.choices[0].message
            
            if assistant_message.tool_calls:
                print(f"  Has {len(assistant_message.tool_calls)} tool call(s)")
            else:
                print(f"  No tool calls - finishing")
            
            if not assistant_message.tool_calls:
                self.conversation_history.append({
                    "role": "assistant",
                    "content": assistant_message.content or ""
                })
                print()
                print(separator)
                print("FINAL ANSWER:")
                print(separator)
                print(assistant_message.content)
                return assistant_message.content
            
            self.conversation_history.append({
                "role": "assistant",
                "content": assistant_message.content or "",
                "tool_calls": [
                    {
                        "id": tc.id,
                        "type": "function",
                        "function": {"name": tc.function.name, "arguments": tc.function.arguments}
                    }
                    for tc in assistant_message.tool_calls
                ]
            })
            
            if assistant_message.content:
                print(f"  Thinking: {assistant_message.content[:80]}...")
            
            for tool_call in assistant_message.tool_calls:
                tool_name = tool_call.function.name
                tool_args = json.loads(tool_call.function.arguments)
                
                print(f"  Executing: {tool_name}({list(tool_args.keys())})")
                
                try:
                    if tool_name == "gitnexus_query":
                        result = gitnexus_query(tool_args["query"])
                    elif tool_name == "gitnexus_context":
                        result = gitnexus_context(tool_args["name"])
                    elif tool_name == "gitnexus_impact":
                        result = gitnexus_impact(
                            tool_args["target"],
                            tool_args.get("direction", "upstream"),
                            tool_args.get("minConfidence", None),
                            tool_args.get("maxDepth", None)
                        )
                    elif tool_name == "gitnexus_cypher":
                        result = gitnexus_cypher(tool_args["query"])
                    elif tool_name == "gitnexus_detect_changes":
                        result = gitnexus_detect_changes(
                            tool_args.get("scope", "staged"),
                            tool_args.get("base_ref", None)
                        )
                    elif tool_name == "gitnexus_rename":
                        result = gitnexus_rename(
                            tool_args["symbol_name"],
                            tool_args["new_name"],
                            tool_args.get("dry_run", True)
                        )
                    elif tool_name == "gitnexus_list_repos":
                        result = gitnexus_list_repos()
                    else:
                        result = {"error": f"Unknown tool: {tool_name}"}
                    
                    result_str = json.dumps(result, indent=2)[:self.max_result_chars]
                    print(f"    Got result ({len(result_str)} chars)")
                    
                except Exception as e:
                    result_str = f"Error: {str(e)}"
                    print(f"    Error: {e}")
                
                self.conversation_history.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result_str
                })
        
        # Max iterations reached — ask the model to conclude from what it gathered
        print()
        print(separator)
        print(f"Max iterations ({self.max_iterations}) reached \u2014 drafting conclusion from gathered information...")
        print(separator)
        print()
        
        try:
            response = client.chat.completions.create(
                model=self.model,
                messages=self.conversation_history + [
                    {
                        "role": "user",
                        "content": (
                            "You have reached the maximum number of research iterations. "
                            "Based on all the information you have gathered so far, provide your best "
                            "answer to the original question. Where your information is incomplete, "
                            "say so clearly rather than speculating."
                        )
                    }
                ],
                temperature=0.5
            )
            conclusion = response.choices[0].message.content or "No conclusion could be generated."
        except Exception as e:
            conclusion = f"Error generating conclusion: {e}"
        
        print(conclusion)
        return conclusion

print("CodebaseAgent defined")


CodebaseAgent defined


## Usage - Ask Your Question Here

In [10]:
# Initialize the agent with better defaults
# Note: gpt-4o-mini was changed to gpt-4-turbo for better reasoning and fewer wasted iterations
agent = CodebaseAgent(model="gpt-4.1-mini", max_iterations=10, max_result_chars=50000)

# ============================================================================
# SPECIFY YOUR QUESTION HERE
# ============================================================================
question = "Which services depend on PaymentService and what would break if we changed its API?"
# question = "Explain the architecture of the microservices-demo repo. What are the main services and how do they interact?"
# question = "What are the most common patterns used in error handling across the codebase?"
# question = "Is it safe to refactor CartService? What services depend on it? Check the usage in those dependencies"
# question = "Show me the checkout flow - which services are called in what order?"
# ============================================================================
# Run the agent
answer = agent.run(question)


Question: Which services depend on PaymentService and what would break if we changed its API?
Model: gpt-4.1-mini

[Iteration 1] Calling OpenAI...
  Has 1 tool call(s)
  Executing: gitnexus_query(['query'])
    Got result (5489 chars)
[Iteration 2] Calling OpenAI...
  Has 1 tool call(s)
  Executing: gitnexus_impact(['target', 'direction'])
    Got result (987 chars)
[Iteration 3] Calling OpenAI...
  No tool calls - finishing

FINAL ANSWER:
The PaymentService class is defined in the file src/recommendationservice/demo_pb2_grpc.py. Two files directly depend on it and would break if its API changed: src/recommendationservice/recommendation_server.py and src/recommendationservice/client.py. The overall risk level of changing PaymentService's API is low since only these two files are directly impacted and no entire processes or modules are affected.


## Conversation History

In [11]:
# Display the conversation history for debugging/understanding
import json

print("\nConversation History:")
print("="*70)
for i, msg in enumerate(agent.conversation_history, 1):
    role = msg.get("role", "unknown").upper()
    content = msg.get("content", "")
    
    if isinstance(content, str):
        preview = content[:200] + "..." if len(content) > 200 else content
        print(f"{i}. [{role}] {preview}")
    elif isinstance(content, list):
        print(f"{i}. [{role}] Tool results ({len(content)} items)")
    else:
        print(f"{i}. [{role}] {str(content)[:100]}")
    print()


Conversation History:
1. [SYSTEM] You are an expert code analysis agent with access to GitNexus tools that query a knowledge graph of the microservices-demo codebase (3477 symbols, 5994 relationships, 33 execution flows).

## Availabl...

2. [USER] Which services depend on PaymentService and what would break if we changed its API?

3. [ASSISTANT] 

4. [TOOL] {
  "processes": [
    {
      "id": "proc_13_main",
      "summary": "Main \u00e2\u2020\u2019 AdServiceImpl",
      "priority": 0.092,
      "symbol_count": 1,
      "process_type": "cross_community"...

5. [ASSISTANT] 

6. [TOOL] {
  "target": {
    "id": "Class:src/recommendationservice/demo_pb2_grpc.py:PaymentService",
    "name": "PaymentService",
    "type": "Class",
    "filePath": "src/recommendationservice/demo_pb2_grpc...

7. [ASSISTANT] The PaymentService class is defined in the file src/recommendationservice/demo_pb2_grpc.py. Two files directly depend on it and would break if its API changed: src/recommendationservice/

## Agent Architecture Overview

```
┌─────────────────────────────────────────────────────────────────┐
│                     User Question                                 │
└────────────────────────────┬────────────────────────────────────┘
                             │
                             ▼
┌─────────────────────────────────────────────────────────────────┐
│                  OpenAI Chat Completion                          │
│              (gpt-4o with tool calling)                         │
└────────────────────────────┬────────────────────────────────────┘
                             │
                    ┌────────┴────────┐
                    ▼                 ▼
         ┌──────────────────┐  ┌──────────────────┐
         │  Reasoning Text  │  │   Tool Calls     │
         └──────────────────┘  └────────┬─────────┘
                                        │
                    ┌───────────────────┼───────────────────┐
                    ▼                   ▼                   ▼
          ┌──────────────────┐  ┌──────────────────┐  ┌──────────────────┐
          │ gitnexus_query   │  │gitnexus_context  │  │ gitnexus_impact  │
          │    (Search)      │  │  (Symbol Info)   │  │  (Blast Radius)  │
          └────────┬─────────┘  └────────┬─────────┘  └────────┬─────────┘
                   │                     │                     │
                   └─────────────────────┼─────────────────────┘
                                         │
                                         ▼
                          ┌──────────────────────────┐
                          │   GitNexus Code Graph    │
                          │   (261 files, 2128      │
                          │    symbols, 5089 edges) │
                          └──────────────────────────┘
                                         │
                    ┌────────────────────┼────────────────────┐
                    ▼                    ▼                    ▼
             ┌──────────────┐    ┌──────────────┐    ┌──────────────┐
             │ Search Results│   │ Call Graph   │    │Process Flows │
             │ & Processes   │   │& Dependencies│    │& Execution  │
             └──────────────┘    └──────────────┘    └──────────────┘
                    │                    │                    │
                    └────────────────────┼────────────────────┘
                                         │
                                         ▼
                          ┌──────────────────────────┐
                          │   Agent Reasoning Loop   │
                          │  (refine → tool calls →  │
                          │   analyze → repeat)      │
                          └──────────────────────────┘
                                         │
                                         ▼
                          ┌──────────────────────────┐
                          │  Final Answer to User    │
                          │ (with code locations &   │
                          │  explanations)           │
                          └──────────────────────────┘
```